# 09 — XGBoost

**Algorithm:** XGBoost (eXtreme Gradient Boosting)  
**Features:** TF-IDF (50K unigrams + bigrams, sparse)  
XGBoost extends gradient boosting with regularisation, column subsampling, and efficient sparse-matrix support.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import (setup_paths, evaluate_and_print, plot_confusion_matrix,
                   plot_roc_curve, plot_pr_curve, plot_learning_curve)
from sklearn.metrics import accuracy_score, log_loss
BASE_DIR, FIGURES_DIR = setup_paths()

try:
    import xgboost as xgb
    print(f'XGBoost version: {xgb.__version__}')
except ImportError:
    raise ImportError('XGBoost is not installed. Run: pip install xgboost')

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True, min_df=2, max_df=0.95)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## Model Training with Eval Log

In [ ]:
from xgboost import XGBClassifier
MODEL_NAME = 'XGBoost'
eval_log = {}
model = XGBClassifier(
    n_estimators=100, random_state=42, eval_metric='logloss',
    n_jobs=-1, verbosity=0,
    early_stopping_rounds=10,
    callbacks=[xgb.callback.EvaluationMonitor(period=10)]
)
t0 = time.time()
model.fit(X_tr, y_train,
          eval_set=[(X_tr, y_train), (X_te, y_test)],
          verbose=False)
train_time = time.time() - t0
print(f'Training completed in {train_time:.2f}s')
print(f'Best iteration: {model.best_iteration}')

In [ ]:
metrics, y_pred, y_prob = evaluate_and_print(model, X_te, y_test, MODEL_NAME, train_time)

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

## Training Convergence: Log-Loss per Round

In [ ]:
results = model.evals_result()
tr_loss = results['validation_0']['logloss']
te_loss = results['validation_1']['logloss']
iterations = np.arange(1, len(tr_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(iterations, tr_loss, color='#3498db', lw=2, label='Train Log-Loss')
axes[0].plot(iterations, te_loss, color='#e74c3c', lw=2, label='Test Log-Loss')
if model.best_iteration:
    axes[0].axvline(model.best_iteration, color='green', linestyle='--', lw=1.5,
                    label=f'Best iter = {model.best_iteration}')
axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('Log-Loss')
axes[0].set_title('XGBoost — Log-Loss Convergence')
axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3)

# Accuracy convergence
staged_tr_acc = [accuracy_score(y_train, (model.predict_proba(X_tr, iteration_range=(0,i+1))[:,1]>=0.5).astype(int))
                 for i in range(0, min(len(tr_loss), 20), 2)]
print('Convergence plot generated.')
axes[1].plot(np.arange(1, len(tr_loss)+1), np.interp(np.arange(1,len(tr_loss)+1),
             np.arange(1,len(tr_loss)+1), 1 - np.array(tr_loss)/max(tr_loss)),
             color='#3498db', lw=2, label='Train (scaled)')
axes[1].plot(np.arange(1, len(te_loss)+1), np.interp(np.arange(1,len(te_loss)+1),
             np.arange(1,len(te_loss)+1), 1 - np.array(te_loss)/max(te_loss)),
             color='#e74c3c', lw=2, label='Test (scaled)')
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('1 - Normalised Log-Loss')
axes[1].set_title('XGBoost — Relative Performance per Round')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.suptitle('XGBoost — Training Convergence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'xgboost_convergence.png'), dpi=150, bbox_inches='tight')
plt.show()

## Feature Importances

In [ ]:
feat_names = np.array(tfidf.get_feature_names_out())
fi = model.feature_importances_
top_idx = np.argsort(fi)[-20:][::-1]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_names[top_idx][::-1], fi[top_idx[::-1]], color='#e67e22', alpha=0.85)
ax.set_xlabel('Feature Importance (F-score)')
ax.set_title('XGBoost — Top 20 Feature Importances')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'xgboost_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

## Learning Curve

In [ ]:
plot_learning_curve(model, X_tr, y_train, X_te, y_test, MODEL_NAME, FIGURES_DIR)